In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!cp -r /content/drive/MyDrive/TechManthan_Internship/Traditional_NLP_Models/* /content/

In [4]:
import torch
from torch import nn, optim
from tqdm import tqdm
from torch.utils.data import DataLoader
from dataset import SequenceDataset
from gru import MultiLayerGRU
from utils import sample

In [5]:
SEQ_LENGTH = 100
HIDDEN_SIZE = 512
NUM_LAYERS = 3
DROPOUT = 0.5

LR = 0.001
BATCH_SIZE = 128
EPOCHS = 1000
DEV = torch.device("cuda")

In [6]:
dataset = SequenceDataset("shakespeare-sonnet.txt", seq_length=SEQ_LENGTH)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
model = MultiLayerGRU(len(dataset.vocab), HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(DEV)
opt = optim.Adam(model.parameters(), lr = LR)
crit = nn.CrossEntropyLoss()

100%|██████████| 94081/94081 [00:01<00:00, 82859.58it/s]


In [ ]:
for e in range(1, EPOCHS + 1):
    loop = tqdm(loader, total=len(loader), leave=True, position=0)
    loop.set_description(f"Epoch : [{e}/{EPOCHS}] | ")
    total_loss = 0
    total_len = 0
    for x, y in loop:
        opt.zero_grad()
        h = torch.zeros((NUM_LAYERS, x.shape[0], HIDDEN_SIZE)).to(DEV)
        yhat, h = model.forward(x.to(DEV), h)
        loss = crit(yhat.view(-1, yhat.shape[-1]), y.view(-1, y.shape[-1]).to(DEV))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5)
        opt.step()

        total_loss += loss.item()
        total_len += 1
        loop.set_postfix(average_loss = total_loss / total_len)

    if e % 10 == 0:
        model = model.eval()
        with torch.no_grad():
            print(f"\n{'=' * 50}\nSample output: \n{sample(model, dataset, 'thou', HIDDEN_SIZE, 400, DEV, NUM_LAYERS)}\n{'=' * 50}\n")

Epoch : [1/1000] | : 100%|██████████| 8/8 [00:06<00:00,  1.21it/s, average_loss=3.24]
Epoch : [2/1000] | : 100%|██████████| 8/8 [00:05<00:00,  1.46it/s, average_loss=2.99]
Epoch : [3/1000] | : 100%|██████████| 8/8 [00:05<00:00,  1.35it/s, average_loss=2.85]
Epoch : [4/1000] | : 100%|██████████| 8/8 [00:05<00:00,  1.45it/s, average_loss=2.68]
Epoch : [5/1000] | : 100%|██████████| 8/8 [00:05<00:00,  1.43it/s, average_loss=2.52]
Epoch : [6/1000] | : 100%|██████████| 8/8 [00:05<00:00,  1.40it/s, average_loss=2.4]
Epoch : [7/1000] | : 100%|██████████| 8/8 [00:05<00:00,  1.45it/s, average_loss=2.32]
Epoch : [8/1000] | : 100%|██████████| 8/8 [00:05<00:00,  1.38it/s, average_loss=2.26]
Epoch : [9/1000] | : 100%|██████████| 8/8 [00:05<00:00,  1.45it/s, average_loss=2.21]
Epoch : [10/1000] | : 100%|██████████| 8/8 [00:05<00:00,  1.38it/s, average_loss=2.16]



Sample output: 
thou hak wo; thet sheime ald noters afrefer lvis,,
the crin whet, and faill ethe the shask thel seill a, aod laking ird thel wire as ure.

aevt, wimt tyeu, sfadinof hor averunine framsury mes, me e lof f ofmareost geirs sevh mey beet foue
bfearseof of jh kee thened to sons de in ton;
fre of sjorme, of saareb yoult fit, youre,
an hiigh tay nou bate thare wiartest coupsed in ofn th ish do pears trade;
bo



Epoch : [11/1000] | : 100%|██████████| 8/8 [00:05<00:00,  1.44it/s, average_loss=2.03]
Epoch : [12/1000] | : 100%|██████████| 8/8 [00:05<00:00,  1.39it/s, average_loss=1.97]
Epoch : [13/1000] | : 100%|██████████| 8/8 [00:05<00:00,  1.43it/s, average_loss=1.93]
Epoch : [14/1000] | : 100%|██████████| 8/8 [00:05<00:00,  1.41it/s, average_loss=1.88]
Epoch : [15/1000] | : 100%|██████████| 8/8 [00:05<00:00,  1.39it/s, average_loss=1.83]
Epoch : [16/1000] | :  75%|███████▌  | 6/8 [00:04<00:01,  1.37it/s, average_loss=1.79]

In [ ]:
torch.save(model.state_dict(), "gru-weights-final.pth")